# Ensemble: DeBERTa + BGE Triplet Model
Combines base.ipynb (DeBERTa classifier) with speedtriplet.ipynb (BGE embeddings) using rank-averaged ensemble with optimized weights

In [ ]:
import os
import gc
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import random
import warnings
from itertools import product
from scipy.stats import rankdata

from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from tqdm import tqdm

# Triplet model imports
from datasets import Dataset as HFDataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    models
)
from sentence_transformers.losses import TripletLoss
import re
from urllib.parse import urlparse

warnings.filterwarnings('ignore')

## CONFIG

In [ ]:
SEED = 42
NFOLDS = 5
MAX_LEN = 256//2
BATCH_SIZE = 16*2
EPOCHS = 5*2
MODEL_PATH = "/kaggle/input/deberta-v3-base/transformers/default/1/deberta-v3-base"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Set seeds
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.cuda.manual_seed_all(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("Using device:", DEVICE)

## Load Data

In [ ]:
train_path = "/kaggle/input/jigsaw-agile-community-rules/train.csv"
test_path = "/kaggle/input/jigsaw-agile-community-rules/test.csv"
sample_sub_path = "/kaggle/input/jigsaw-agile-community-rules/sample_submission.csv"

df = pd.read_csv(train_path)
df["text"] = df["rule"] + " [SEP] " + df["body"]
df["label"] = df["rule_violation"].astype(float)

test_df = pd.read_csv(test_path)

In [ ]:
def add_data(dataframe):
    ret=[[],[]]
    for i in ['positive_example_1','positive_example_2','negative_example_1','negative_example_2']:
        tmp= (dataframe['rule']+' [SEP] '+ dataframe[i]).tolist()
        ret[0]+= tmp
        ret[1]+= [1]*len(tmp) if 'positive' in i else [0]*len(tmp)
    return ret

# Get augmented data from both df and test_df
augmented_train = add_data(df)
augmented_test = add_data(test_df)

# Combine original df with augmented data
augmented_texts =  df.text.tolist()+augmented_train[0] + augmented_test[0]
augmented_labels =  df.label.tolist()+augmented_train[1] + augmented_test[1]

# Create new augmented dataframe
augmented_df = pd.DataFrame({
    'text': augmented_texts,
    'label': augmented_labels
})
print(f'Before:{augmented_df.shape}')
augmented_df = augmented_df.groupby(augmented_df['text'].str.lower(), as_index=False).agg({
    'text': 'first',
    'label': 'mean'
})
print('After:',augmented_df.shape)
augmented_df['rule']= augmented_df.text.apply(lambda x: x.split(' [SEP] ')[0])
augmented_df['body']= augmented_df.text.apply(lambda x: x.split(' [SEP] ')[1])

rule_map= {i:j for j,i in enumerate(augmented_df.rule.str.lower().unique())}
augmented_df['rule_id']= augmented_df.rule.str.lower().map(rule_map)

augmented_df.head()

## Model 1: DeBERTa Base Model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast = False)

In [ ]:
class JigsawDataset(Dataset):
    def __init__(self, texts, labels, rule_ids, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.rule_ids = rule_ids

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        enc = self.tokenizer(
            text, padding='max_length', truncation=True, max_length=self.max_len, return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        item['rule_ids']= torch.tensor(self.rule_ids[idx])
        return item

In [ ]:
class JigsawModel(nn.Module):
    def __init__(self, model_path):
        super().__init__()
        self.base = AutoModel.from_pretrained(model_path)
        self.drop = nn.Dropout(0.15)
        self.out = nn.Linear(self.base.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.base(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]
        return self.out(self.drop(pooled)).squeeze(1)

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0
    for batch in tqdm(loader):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        logits = model(input_ids, mask)
        loss = nn.BCEWithLogitsLoss()(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        optimizer.step()
        if scheduler:
            scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)

In [ ]:
def validate(model, loader):
    model.eval()
    preds, targets, rule_ids_list = [], [], []
    total_loss = 0
    criterion = nn.BCEWithLogitsLoss()
    
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            rule_ids = batch["rule_ids"]
            
            logits = model(input_ids, mask)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            
            preds.extend(torch.sigmoid(logits).cpu().numpy())
            targets.extend(labels.cpu().numpy())
            rule_ids_list.extend(rule_ids.cpu().numpy() if torch.is_tensor(rule_ids) else rule_ids)
    
    preds = np.array(preds)
    targets = np.array(targets)
    rule_ids_array = np.array(rule_ids_list)
    
    unique_rules = np.unique(rule_ids_array)
    rule_aucs = {}
    
    for rule_id in unique_rules:
        rule_mask = rule_ids_array == rule_id
        rule_preds = preds[rule_mask]
        rule_targets = targets[rule_mask]
        
        if len(np.unique(rule_targets >= 0.5)) > 1:
            rule_auc = roc_auc_score(rule_targets >= 0.5, rule_preds)
            rule_aucs[rule_id] = rule_auc
        else:
            rule_aucs[rule_id] = np.nan
    
    valid_aucs = [auc for auc in rule_aucs.values() if not np.isnan(auc)]
    avg_auc_per_rule = np.mean(valid_aucs) if valid_aucs else 0
    
    val_loss = total_loss / len(loader)
    return avg_auc_per_rule, val_loss, preds, targets

## Train Base Model or Get OOF Predictions

In [ ]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Training phase
    folds = StratifiedKFold(n_splits=NFOLDS, shuffle=True, random_state=SEED)
    for fold, (tr_idx, val_idx) in enumerate(folds.split(augmented_df, augmented_df["rule"])):
        print('--------- ','FOLD: ',fold,' --------')
        all_preds = []
        val_ds = JigsawDataset(
            augmented_df.iloc[val_idx]['text'].tolist(), 
            augmented_df.iloc[val_idx]['label'].tolist(), 
            augmented_df.iloc[val_idx]['rule_id'].tolist(), 
            tokenizer, MAX_LEN
        )
    
        train_ds = JigsawDataset(
            augmented_df.iloc[tr_idx]['text'].tolist(), 
            augmented_df.iloc[tr_idx]['label'].tolist(), 
            augmented_df.iloc[tr_idx]['rule_id'].tolist(), 
            tokenizer, MAX_LEN,
        )
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
    
        model = JigsawModel(MODEL_PATH).to(DEVICE)
        for name, param in model.named_parameters():
            if name.startswith('base.embedding'):
                param.requires_grad = False
                
        print('Trainable Params: ',sum(i.numel() for i in model.parameters() if i.requires_grad))
       
        optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5,eps=1e-6)
        total_steps= EPOCHS*len(train_loader)
        warmup_steps= 0.1*total_steps
        scheduler = get_linear_schedule_with_warmup(
                optimizer,
                num_warmup_steps=warmup_steps,
                num_training_steps=total_steps,
            )

        best_auc=0
        for epoch in range(EPOCHS):
            print(f"Epoch {epoch+1}/{EPOCHS}")
            loss = train_one_epoch(model, train_loader, optimizer, scheduler)
            val_auc, val_loss, val_preds, val_targets = validate(model, val_loader)
            
            print(f"Loss: {loss:.4f}, Val Loss: {val_loss:.4f}, Val AUC: {val_auc:.4f}")
            if val_auc > best_auc:
                best_auc = val_auc
                torch.save(model.state_dict(), f"model_fold{fold}_auc.bin")
    
        all_preds.append(pd.Series(val_preds))

## Get Base Model OOF Predictions on Train

In [ ]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    print("=== Getting Base Model OOF Predictions ===")
    
    oof = []
    val_id = []
    folds = StratifiedKFold(n_splits=NFOLDS, shuffle=True, random_state=SEED)
    
    for fold, (tr_idx, val_idx) in enumerate(folds.split(augmented_df, augmented_df["rule"])):
        print(f'--------- FOLD {fold} (OOF Recovery) --------')
        torch.cuda.empty_cache()
        gc.collect()
        
        val_ds = JigsawDataset(
            augmented_df.iloc[val_idx]['text'].tolist(), 
            augmented_df.iloc[val_idx]['label'].tolist(), 
            augmented_df.iloc[val_idx]['rule_id'].tolist(), 
            tokenizer, MAX_LEN
        )
    
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
        model = JigsawModel(MODEL_PATH).to(DEVICE)
        model.load_state_dict(torch.load(f"model_fold{fold}_auc.bin", map_location=DEVICE))
        
        val_auc, val_loss, val_preds, targets = validate(model, val_loader)
        val_id.extend(list(val_idx))
        oof.append((val_preds, targets))
        print(f'Fold {fold} AUC: {val_auc:.4f}')

    all_preds_raw = np.concatenate([x[0] for x in oof])
    all_preds = np.zeros_like(all_preds_raw)
    all_preds[val_id] = all_preds_raw
    bert_preds = all_preds
    
    augmented_df['lookup_key'] = augmented_df['text'].str.lower()
    df['lookup_key'] = df['text'].str.lower()
    augmented_df['predictions'] = bert_preds
    
    pred_map = augmented_df.set_index('lookup_key')['predictions'].to_dict()
    df['base_train_preds'] = df['lookup_key'].map(pred_map)
    df = df.drop('lookup_key', axis=1)
    
    print(f"Base model train predictions shape: {df['base_train_preds'].shape}")
    print(f"Non-null predictions: {df['base_train_preds'].notna().sum()}")

## Get Base Model Test Predictions

In [ ]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    print("=== Getting Base Model Test Predictions ===")
    test_df["text"] = test_df["rule"] + " [SEP] " + test_df["body"]
    
    test_preds = []
    for fold in range(NFOLDS):
        model = JigsawModel(MODEL_PATH).to(DEVICE)
        wts = torch.load(f"model_fold{fold}_auc.bin", map_location=DEVICE)
        wts = {k.replace('module.',''):v for k,v in wts.items()}
        model.load_state_dict(wts)
        model.eval()
    
        test_ds = JigsawDataset(test_df['text'].tolist(), [0]*len(test_df), [0]*len(test_df), tokenizer, MAX_LEN)
        test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)
    
        fold_preds = []
        with torch.no_grad():
            for batch in test_loader:
                ids = batch['input_ids'].to(DEVICE)
                mask = batch['attention_mask'].to(DEVICE)
                logits = model(ids, mask)
                fold_preds.extend(torch.sigmoid(logits).cpu().numpy())
        test_preds.append(fold_preds)
    
    base_test_preds = np.mean(test_preds, axis=0)
    print(f"Base model test predictions shape: {base_test_preds.shape}")

## Model 2: Triplet BGE Model

In [ ]:
def cleaner(text):
    """Replace URLs with format: <url>: (domain/important-path)"""
    if not text:
        return text

    url_pattern = r'https?://[^\s<>"{}|\\^`\[\]]+'

    def replace_url(match):
        url = match.group(0)
        try:
            parsed = urlparse(url)
            domain = parsed.netloc.lower()
            if domain.startswith('www.'):
                domain = domain[4:]

            path_parts = [part for part in parsed.path.split('/') if part]
            if path_parts:
                important_path = '/'.join(path_parts[:2])
                return f"<url>: ({domain}/{important_path})"
            else:
                return f"<url>: ({domain})"
        except:
            return "<url>: (unknown)"

    return re.sub(url_pattern, replace_url, str(text))

In [ ]:
def collect_all_texts(test_df):
    """Collect all unique texts from test set."""
    print("\nCollecting all texts for embedding...")
    
    all_texts = set()
    
    for body in test_df['body']:
        if pd.notna(body):
            all_texts.add(cleaner(str(body)))
    
    example_cols = ['positive_example_1', 'positive_example_2', 
                   'negative_example_1', 'negative_example_2']
    
    for col in example_cols:
        for example in test_df[col]:
            if pd.notna(example):
                all_texts.add(cleaner(str(example)))
    
    all_texts = list(all_texts)
    print(f"Collected {len(all_texts)} unique texts")
    return all_texts

In [ ]:
def generate_embeddings(texts, model, batch_size=64):
    """Generate BGE embeddings for all texts."""
    print(f"Generating embeddings for {len(texts)} texts...")
    
    embeddings = model.encode(
        sentences=texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_tensor=False,
        normalize_embeddings=True
    )
    
    return embeddings

In [ ]:
def create_test_triplet_dataset(test_df, augmentation_factor=2, random_seed=42, subsample_fraction=1.0):
    """Create triplet dataset from test data."""
    random.seed(random_seed)
    np.random.seed(random_seed)
    
    anchors = []
    positives = []
    negatives = []
    
    print("Creating rule-aligned triplets from test data...")
    
    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Processing test rows"):
        rule = cleaner(str(row['rule']))
        
        pos_examples = []
        neg_examples = []

        for neg_col in ['negative_example_1', 'negative_example_2']:
            if pd.notna(row[neg_col]):
                pos_examples.append(cleaner(str(row[neg_col])))

        for pos_col in ['positive_example_1', 'positive_example_2']:
            if pd.notna(row[pos_col]):
                neg_examples.append(cleaner(str(row[pos_col])))
        
        for pos_ex in pos_examples:
            for neg_ex in neg_examples:
                anchors.append(rule)
                positives.append(pos_ex)
                negatives.append(neg_ex)
    
    if augmentation_factor > 0:
        print(f"Adding {augmentation_factor}x augmentation...")
        
        rule_positives = {}
        rule_negatives = {}
        
        for rule in test_df['rule'].unique():
            rule_df = test_df[test_df['rule'] == rule]
            
            pos_pool = []
            neg_pool = []
            
            for _, row in rule_df.iterrows():
                for neg_col in ['negative_example_1', 'negative_example_2']:
                    if pd.notna(row[neg_col]):
                        pos_pool.append(cleaner(str(row[neg_col])))
                for pos_col in ['positive_example_1', 'positive_example_2']:
                    if pd.notna(row[pos_col]):
                        neg_pool.append(cleaner(str(row[pos_col])))
            
            rule_positives[rule] = list(set(pos_pool))
            rule_negatives[rule] = list(set(neg_pool))
        
        for rule in test_df['rule'].unique():
            clean_rule = cleaner(str(rule))
            pos_pool = rule_positives[rule]
            neg_pool = rule_negatives[rule]
            
            n_samples = min(augmentation_factor * len(pos_pool), len(pos_pool) * len(neg_pool))
            
            for _ in range(n_samples):
                if pos_pool and neg_pool:
                    anchors.append(clean_rule)
                    positives.append(random.choice(pos_pool))
                    negatives.append(random.choice(neg_pool))
    
    combined = list(zip(anchors, positives, negatives))
    random.shuffle(combined)
    
    original_count = len(combined)
    if subsample_fraction < 1.0:
        n_samples = int(len(combined) * subsample_fraction)
        combined = combined[:n_samples]
        print(f"Subsampled {original_count} -> {len(combined)} triplets ({subsample_fraction*100:.1f}%)")
    
    anchors, positives, negatives = zip(*combined) if combined else ([], [], [])
    
    print(f"Created {len(anchors)} triplets from test data")
    
    dataset = HFDataset.from_dict({
        'anchor': list(anchors),
        'positive': list(positives),
        'negative': list(negatives)
    })
    
    return dataset

In [ ]:
def fine_tune_model(model, train_dataset, epochs=1, batch_size=32, learning_rate=2e-5, margin=0.25, output_dir="./models/test-finetuned-bge"):
    """Fine-tune the sentence transformer model using triplet loss."""
    
    print(f"Fine-tuning model on {len(train_dataset)} triplets...")
    
    loss = TripletLoss(model=model, triplet_margin=margin)
    
    dataset_size = len(train_dataset)
    steps_per_epoch = max(1, dataset_size // batch_size)
    max_steps = steps_per_epoch * epochs

    args = SentenceTransformerTrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        warmup_steps=0,
        learning_rate=learning_rate,
        logging_steps=max(1, max_steps // 4),
        save_strategy="epoch",
        save_total_limit=1,
        fp16=True,
        max_grad_norm=1.0,
        dataloader_drop_last=False,
        gradient_checkpointing=True,
        gradient_accumulation_steps=1,
        max_steps=max_steps,
        report_to="none"
    )
    
    trainer = SentenceTransformerTrainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        loss=loss,
    )
    
    trainer.train()
    
    final_model_path = f"{output_dir}/final"
    print(f"Saving fine-tuned model to {final_model_path}...")
    model.save_pretrained(final_model_path)
    
    return model, final_model_path

In [ ]:
def load_or_create_finetuned_model(test_df):
    """Load fine-tuned model if exists, otherwise create and fine-tune it."""
    
    fine_tuned_path = "./models/test-finetuned-bge/final"
    
    if os.path.exists(fine_tuned_path):
        print(f"Loading existing fine-tuned model from {fine_tuned_path}...")
        try:
            word_embedding_model = models.Transformer(fine_tuned_path, max_seq_length=128, do_lower_case=True)
            pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension(), pooling_mode="mean")
            model = SentenceTransformer(modules=[word_embedding_model, pooling_model])
            print("Loaded fine-tuned model with explicit pooling")
        except:
            model = SentenceTransformer(fine_tuned_path)
            print("Loaded fine-tuned model with default configuration")
        model.half()
        return model
    
    print("Fine-tuned model not found. Creating new one...")
    
    print("Loading base BGE embedding model...")
    try:
        model_path = "/kaggle/input/baai/transformers/bge-base-en-v1.5/1"
        word_embedding_model = models.Transformer(model_path, max_seq_length=128, do_lower_case=True)
        pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension(), pooling_mode="mean")
        base_model = SentenceTransformer(modules=[word_embedding_model, pooling_model])
        print("Loaded base model from Kaggle path with explicit pooling")
    except:
        model_path = "BAAI/bge-base-en-v1.5"
        word_embedding_model = models.Transformer(model_path, max_seq_length=128, do_lower_case=True)
        pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension(), pooling_mode="mean")
        base_model = SentenceTransformer(modules=[word_embedding_model, pooling_model])
        print("Loaded base model from HuggingFace with explicit pooling")
    
    triplet_dataset = create_test_triplet_dataset(test_df, augmentation_factor=2, subsample_fraction=1.)
    
    fine_tuned_model, model_path = fine_tune_model(
        model=base_model,
        train_dataset=triplet_dataset,
        epochs=1,
        batch_size=32,
        learning_rate=2e-5,
        margin=0.25
    )
    
    print(f"Fine-tuning completed. Model saved to: {model_path}")
    fine_tuned_model.half()
    return fine_tuned_model

In [ ]:
def generate_rule_embeddings(test_df, model):
    """Generate embeddings for each unique rule."""
    print("Generating rule embeddings...")
    
    unique_rules = test_df['rule'].unique()
    rule_embeddings = {}
    
    for rule in unique_rules:
        clean_rule = cleaner(str(rule))
        rule_emb = model.encode(
            clean_rule,
            convert_to_tensor=False,
            normalize_embeddings=True
        )
        rule_embeddings[rule] = rule_emb
        
    print(f"Generated embeddings for {len(rule_embeddings)} rules")
    return rule_embeddings

In [ ]:
def create_rule_centroids(test_df, text_to_embedding, rule_embeddings):
    """Create single centroid (mean) for positive and negative examples for each rule."""
    print(f"\nCreating rule centroids (single mean centroid per type)...")

    rule_centroids = {}

    for rule in test_df['rule'].unique():
        rule_data = test_df[test_df['rule'] == rule]

        pos_embeddings = []
        for _, row in rule_data.iterrows():
            for col in ['positive_example_1', 'positive_example_2']:
                if pd.notna(row[col]):
                    clean_text = cleaner(str(row[col]))
                    if clean_text in text_to_embedding:
                        pos_embeddings.append(text_to_embedding[clean_text])

        neg_embeddings = []
        for _, row in rule_data.iterrows():
            for col in ['negative_example_1', 'negative_example_2']:
                if pd.notna(row[col]):
                    clean_text = cleaner(str(row[col]))
                    if clean_text in text_to_embedding:
                        neg_embeddings.append(text_to_embedding[clean_text])

        if pos_embeddings and neg_embeddings:
            pos_embeddings = np.array(pos_embeddings)
            neg_embeddings = np.array(neg_embeddings)

            pos_centroid = pos_embeddings.mean(axis=0)
            neg_centroid = neg_embeddings.mean(axis=0)

            pos_centroid = pos_centroid / np.linalg.norm(pos_centroid)
            neg_centroid = neg_centroid / np.linalg.norm(neg_centroid)

            rule_centroids[rule] = {
                'positive': pos_centroid,
                'negative': neg_centroid,
                'pos_count': len(pos_embeddings),
                'neg_count': len(neg_embeddings),
                'rule_embedding': rule_embeddings[rule]
            }

            print(f"  Rule: {rule[:50]}... - Pos: {len(pos_embeddings)}, Neg: {len(neg_embeddings)}")

    print(f"Created centroids for {len(rule_centroids)} rules")
    return rule_centroids

In [ ]:
def predict_with_triplet(data_df, text_to_embedding, rule_centroids, is_test=True):
    """Predict using Euclidean distance between body and pos/neg centroids."""
    print(f"\nMaking predictions with triplet model ({'test' if is_test else 'train'} set)...")

    row_ids = []
    predictions = []

    for rule in data_df['rule'].unique():
        print(f"  Processing rule: {rule[:50]}...")
        rule_data = data_df[data_df['rule'] == rule]

        if rule not in rule_centroids:
            continue

        pos_centroid = rule_centroids[rule]['positive']
        neg_centroid = rule_centroids[rule]['negative']

        valid_embeddings = []
        valid_row_ids = []

        for _, row in rule_data.iterrows():
            body = cleaner(str(row['body']))
            row_id = row['row_id'] if is_test else row.name

            if body in text_to_embedding:
                valid_embeddings.append(text_to_embedding[body])
                valid_row_ids.append(row_id)

        if not valid_embeddings:
            continue

        query_embeddings = np.array(valid_embeddings)

        pos_distances = np.linalg.norm(query_embeddings - pos_centroid, axis=1)
        neg_distances = np.linalg.norm(query_embeddings - neg_centroid, axis=1)

        rule_predictions = neg_distances - pos_distances

        row_ids.extend(valid_row_ids)
        predictions.extend(rule_predictions)

    print(f"Made predictions for {len(predictions)} examples")
    return row_ids, np.array(predictions)

## Run Triplet Model

In [ ]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    print("\n" + "="*70)
    print("TRIPLET MODEL - TRAINING AND INFERENCE")
    print("="*70)
    
    # Load and fine-tune model on test data
    triplet_model = load_or_create_finetuned_model(test_df)
    
    # Collect all texts from test set
    all_texts = collect_all_texts(test_df)
    all_embeddings = generate_embeddings(all_texts, triplet_model)
    text_to_embedding = {text: emb for text, emb in zip(all_texts, all_embeddings)}
    
    # Generate rule embeddings and centroids from test data
    rule_embeddings = generate_rule_embeddings(test_df, triplet_model)
    rule_centroids = create_rule_centroids(test_df, text_to_embedding, rule_embeddings)

## Get Triplet Model Train Predictions

In [ ]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    print("\n" + "="*70)
    print("TRIPLET MODEL - TRAIN PREDICTIONS")
    print("="*70)
    
    # Add row_id to train data
    train_for_triplet = pd.read_csv(train_path)
    train_for_triplet['row_id'] = range(len(train_for_triplet))
    
    # Collect train texts and embed them
    train_bodies = [cleaner(str(body)) for body in train_for_triplet['body']]
    unique_train_bodies = list(set(train_bodies))
    print(f"Embedding {len(unique_train_bodies)} unique train bodies...")
    
    train_embeddings = generate_embeddings(unique_train_bodies, triplet_model)
    train_text_to_embedding = {text: emb for text, emb in zip(unique_train_bodies, train_embeddings)}
    
    # Update text_to_embedding with train texts
    text_to_embedding.update(train_text_to_embedding)
    
    # Predict on train
    train_row_ids, triplet_train_preds = predict_with_triplet(train_for_triplet, text_to_embedding, rule_centroids, is_test=False)
    
    # Map predictions back to df
    triplet_train_df = pd.DataFrame({'row_id': train_row_ids, 'triplet_train_preds': triplet_train_preds})
    df = df.merge(triplet_train_df, left_index=True, right_on='row_id', how='left')
    
    print(f"Triplet train predictions shape: {df['triplet_train_preds'].shape}")
    print(f"Non-null predictions: {df['triplet_train_preds'].notna().sum()}")

## Get Triplet Model Test Predictions

In [ ]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    print("\n" + "="*70)
    print("TRIPLET MODEL - TEST PREDICTIONS")
    print("="*70)
    
    test_row_ids, triplet_test_preds = predict_with_triplet(test_df, text_to_embedding, rule_centroids, is_test=True)
    
    print(f"Triplet test predictions shape: {len(triplet_test_preds)}")

## Ensemble: Optimize Weights on Train, Apply to Test

In [ ]:
def compute_per_rule_auc(df, pred_col, label_col='label', rule_col='rule'):
    """Compute mean per-rule AUC."""
    aucs = []
    for rule in df[rule_col].unique():
        rule_df = df[df[rule_col] == rule]
        if len(rule_df[label_col].unique()) > 1:
            auc = roc_auc_score(rule_df[label_col] >= 0.5, rule_df[pred_col])
            aucs.append(auc)
    return np.mean(aucs) if aucs else 0

In [ ]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    print("\n" + "="*70)
    print("ENSEMBLE OPTIMIZATION")
    print("="*70)
    
    # Filter valid predictions
    valid_df = df[df['base_train_preds'].notna() & df['triplet_train_preds'].notna()].copy()
    print(f"Valid train samples for optimization: {len(valid_df)}")
    
    # Convert to ranks
    valid_df['base_rank'] = rankdata(valid_df['base_train_preds'])
    valid_df['triplet_rank'] = rankdata(valid_df['triplet_train_preds'])
    
    # Grid search for best weight
    best_auc = 0
    best_weight = 0.5
    
    print("\nGrid searching for optimal ensemble weight...")
    for w in np.arange(0, 1.05, 0.05):
        ensemble_rank = w * valid_df['base_rank'] + (1 - w) * valid_df['triplet_rank']
        valid_df['ensemble_preds'] = ensemble_rank
        auc = compute_per_rule_auc(valid_df, 'ensemble_preds', 'label', 'rule')
        print(f"  Weight {w:.2f}: AUC = {auc:.4f}")
        if auc > best_auc:
            best_auc = auc
            best_weight = w
    
    print(f"\nBest ensemble weight: {best_weight:.2f} (AUC: {best_auc:.4f})")
    print(f"  Base model weight: {best_weight:.2f}")
    print(f"  Triplet model weight: {1-best_weight:.2f}")

## Create Final Submission

In [ ]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    print("\n" + "="*70)
    print("CREATING FINAL SUBMISSION")
    print("="*70)
    
    # Convert test predictions to ranks
    base_test_ranks = rankdata(base_test_preds)
    triplet_test_ranks = rankdata(triplet_test_preds)
    
    # Apply optimal weights
    ensemble_test_ranks = best_weight * base_test_ranks + (1 - best_weight) * triplet_test_ranks
    
    # Create submission
    submission = pd.DataFrame({
        'row_id': test_row_ids,
        'rule_violation': ensemble_test_ranks
    })
    
    submission.to_csv('submission.csv', index=False)
    print(f"✅ Submission saved as submission.csv")
    print(f"Predictions: min={ensemble_test_ranks.min():.4f}, max={ensemble_test_ranks.max():.4f}, mean={ensemble_test_ranks.mean():.4f}")
else:
    !touch submission.csv

In [ ]:
!head -n 4 submission.csv